# Concept

Earlier in Module 1.1, you manually updated model parameters using standard inline subtraction (w.data -= lr * w.grad). While this works for tiny academic examples, manually updating thousands of layered tensors in a deep network is wildly impractical.

To automate this, PyTorch provides the torch.optim package. This engine decouples the optimization algorithm from the model architecture entirely. You pass your model's parameters directly into an optimizer object, and calling .step() automatically updates every single registered weight and bias matrix across your network layout using the gradients computed during backpropagation.<br>
The two most widely used production optimizers are:
<br>1.optim.SGD (Stochastic Gradient Descent): Updates parameters by multiplying gradients by a fixed, uniform learning rate.
<br>2. optim.Adam (Adaptive Moment Estimation): An advanced optimizer that tracks historic gradient moving averages to automatically scale the learning rate dynamically for each individual parameter, leading to significantly faster and more stable training phases.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define a standard structural layer
layer = nn.Linear(3, 1)

# Initialize the Adam optimizer engine and hand over the layer's parameters
optimizer = optim.Adam(layer.parameters(), lr=0.01)

# --- SIMULATED TRAINING STEP CYCLE ---
# Mock forward and backward pass
outputs = layer(torch.randn(2, 3))
loss = torch.mean((outputs - 1.0) ** 2)
loss.backward()

# AUTOMATED UPDATE STEP: Mutates all parameters in layer.parameters() automatically!
optimizer.step()

# AUTOMATED RESET STEP: Replaces manually looping w.grad.zero_()
optimizer.zero_grad()

# Exercise

Write a script that initializes a neural network layer and links it directly to an optimization engine.
<br>1. Instantiate a linear structural layer named dense_layer that accepts 4 input features and scales them to 2 output features.
<br>2. Instantiate an SGD optimizer engine (optim.SGD) linked directly to your dense_layer's parameters. Configure it with a learning rate of 0.05.
<br>3. Generate a mock batch of features named inputs of shape (2, 4). Pass them forward through your layer to create a prediction tensor named preds.
<br>4. Calculate a basic mean squared error scalar loss against a random target tensor of shape (2, 2). Trigger backpropagation.
<br>5. Execute the automated optimization update step, followed by the automated gradient clearing reset step using the optimizer's built-in methods.

In [7]:
dense_layer = nn.Linear(in_features=4, out_features=2)

optimizer = optim.SGD(dense_layer.parameters(), lr=0.05)

mock_input = torch.rand(2,4)
preds = dense_layer(mock_input)
target = torch.rand(2,2)

loss_fn = nn.MSELoss()
loss = loss_fn(preds, target)
loss.backward()

optimizer.step()
optimizer.zero_grad()


# Concept

When your network contains multiple independent modules or sub-networks (for example, an Actor-Critic architecture in Reinforcement Learning, or a Generator and Discriminator setup in a GAN), passing your entire architecture's parameters to a single optimizer forces a uniform learning rate across everything.

To solve this, torch.optim allows you to pass a list of per-parameter options (dictionaries) to the constructor. This lets you assign unique, fine-tuned learning rates or weight decay factors to different sub-modules within the exact same optimization engine.

In [ ]:
 model_head = nn.Linear(10, 5)
model_backbone = nn.Linear(5, 2)

# Pass structured dictionaries specifying isolated learning rates per module
optimizer = optim.Adam([
    {"params": model_backbone.parameters(), "lr": 0.001}, # Slow adjustments for feature extraction
    {"params": model_head.parameters(), "lr": 0.01}       # Faster adjustments for final output classification
])